In [ ]:
import argparse 
import numpy as np 
import os 
import pandas as pd 
import logging
from pathlib import Path
from tqdm import tqdm

def process_dirname(dirname):
    SPL = dirname.split("_")
    GAS = SPL[0]
    TEMP = SPL[1]
    LOW = SPL[2]
    HIGH = SPL[4]
    INPUT = SPL[6]
    SAMPLING = "_".join(SPL[8:-2])
    MODEL = SPL[-5]
    return {
        "GAS": GAS,
        "TEMP": TEMP,
        "LOW": LOW,
        "HIGH": HIGH,
        "INPUT": INPUT,
        "SAMPLING": SAMPLING,
        "MODEL": MODEL,
    }

# logging 설정
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

S_PATH = Path("./")
N_TRIAL = 5
LIS = []

# 상위 디렉토리 순회
for P in tqdm([x for x in os.listdir(S_PATH) if os.path.isdir(S_PATH/x)], desc="Processing directories"):
    DICT = process_dirname(P)

    TRIALS = [f"trial_{i:03d}" for i in range(1, N_TRIAL + 1)]
    R2_LIS, MAE_LIS, MAPE_LIS = [], [], []

    for TRIAL in tqdm(TRIALS, desc=f"Trials for {P}", leave=False):
        T = TRIAL.replace("_", "")
        CSV = S_PATH / P / TRIAL / f"metrics_holdout_{T}.csv"
        if not CSV.exists():
            continue

        df = pd.read_csv(CSV)
        # 예시 CSV 컬럼:
        # Model,R2,MAE,RMSE,MAPE_percent,n_train,n_test,fit_time_sec,predict_time_sec

        R2 = df["R2"].values[0]
        MAE = df["MAE"].values[0]
        MAPE = df["MAPE_percent"].values[0]

        R2_LIS.append(R2)
        MAE_LIS.append(MAE)
        MAPE_LIS.append(MAPE)

    if R2_LIS:
        DICT["R2_MEAN"] = np.mean(R2_LIS)
        DICT["R2_STD"] = np.std(R2_LIS)
        DICT["MAE_MEAN"] = np.mean(MAE_LIS)
        DICT["MAE_STD"] = np.std(MAE_LIS)
        DICT["MAPE_MEAN"] = np.mean(MAPE_LIS)
        DICT["MAPE_STD"] = np.std(MAPE_LIS)
        LIS.append(DICT)

out_df = pd.DataFrame(LIS)
out_df.to_csv("METRICS_OUTPUT.csv", index=False)


Processing directories: 100%|██████████| 819/819 [00:17<00:00, 47.74it/s]


In [ ]:
df

,Model,R2,MAE,RMSE,MAPE_percent,n_train,n_test,fit_time_sec,predict_time_sec
0,cat,0.227283,0.331947,0.756362,31.434057,5684,1421,5.063227,0.003128


In [ ]:
import pandas as pd
import logging
from tqdm import tqdm

def process_metrics(input_file: str, output_file: str):
    # 로깅 설정
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        handlers=[logging.StreamHandler()]
    )

    # 데이터 불러오기
    df = pd.read_csv(input_file)
    logging.info(f"Data loaded with {len(df)} rows and {len(df.columns)} columns from {input_file}")

    # 정렬 기준
    sort_cols = ["GAS", "TEMP", "MODEL", "SAMPLING", "HIGH", "LOW"]

    # 정렬 (tqdm 사용)
    for col in tqdm(sort_cols, desc="Sorting step-by-step"):
        df = df.sort_values(by=col, kind="mergesort")
        logging.info(f"Sorted by {col}")

    df_sorted = df.sort_values(by=sort_cols, ascending=True, kind="mergesort").reset_index(drop=True)
    logging.info("Final sorting completed.")

    # HIGH를 컬럼으로 pivot
    df_pivot = df_sorted.pivot(
        index=["GAS", "TEMP", "MODEL", "SAMPLING", "LOW"],
        columns="HIGH",
        values=[
            "R2_MEAN", "R2_STD",
            "MAE_MEAN", "MAE_STD",
            "MAPE_MEAN", "MAPE_STD",  # ✅ 추가
            "INPUT"
        ]
    )

    # 컬럼명 정리
    df_pivot.columns = [f"{val}_HIGH_{col}" for val, col in df_pivot.columns]
    df_pivot = df_pivot.reset_index()

    # 저장
    df_pivot.to_csv(output_file, index=False)
    logging.info(f"Saved pivoted results to {output_file}")

# 실행
process_metrics("METRICS_OUTPUT.csv", "METRICS_OUTPUT_pivot.csv")


2025-09-14 05:27:23,746 [INFO] Data loaded with 819 rows and 13 columns from METRICS_OUTPUT.csv
Sorting step-by-step:   0%|          | 0/6 [00:00<?, ?it/s]2025-09-14 05:27:23,748 [INFO] Sorted by GAS
2025-09-14 05:27:23,749 [INFO] Sorted by TEMP
2025-09-14 05:27:23,750 [INFO] Sorted by MODEL
2025-09-14 05:27:23,751 [INFO] Sorted by SAMPLING
2025-09-14 05:27:23,752 [INFO] Sorted by HIGH
2025-09-14 05:27:23,753 [INFO] Sorted by LOW
Sorting step-by-step: 100%|██████████| 6/6 [00:00<00:00, 999.95it/s]
2025-09-14 05:27:23,757 [INFO] Final sorting completed.
2025-09-14 05:27:23,772 [INFO] Saved pivoted results to METRICS_OUTPUT_pivot.csv
